# Fine-Tuning Llama3-8B for Wordpress QA

## Install & Import Libraries

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset
import torch

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("ggplot")

%matplotlib inline

## Preprocessing Datasets

### Load Datasets

In [ ]:
base_data = pd.read_json("hf://datasets/Amod/mental_health_counseling_conversations/combined_dataset.json", lines=True)
my_data = pd.read_json("/data/wordpress_faq.json", orient="index")
print(my_data.head(3), base_data.head(3), sep="\n")

### Base Data Context Distribution

In [ ]:
base_data["Context_length"] = base_data["Context"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(base_data["Context_length"], bins=50, kde=True)
plt.title("Distribution of Context Lengths")
plt.xlabel("Length of Context")
plt.ylabel("Frequency")

plt.show()

In [ ]:
filtered_base_data = base_data[base_data["Context_length"] <= 1500]

ln_Context = filtered_base_data["Context"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(ln_Context, bins=50, kde=True)
plt.title("Distribution of Context Lengths")
plt.xlabel("Length of Context")
plt.ylabel("Frequency")

plt.show()

### My Data Question Distribution

In [ ]:
my_data["question_length"] = my_data["question"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(my_data["question_length"], bins=50, kde=True)
plt.title("Distribution of Question Lengths")
plt.xlabel("Length of Question")
plt.ylabel("Frequency")

plt.show()

In [ ]:
filtered_my_data = my_data[my_data["question_length"] <= 100]

ln_question = filtered_my_data["question"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(ln_question, bins=50, kde=True)
plt.title("Distribution of Question Lengths")
plt.xlabel("Length of Question")
plt.ylabel("Frequency")

plt.show()

### Base Data Response Distribution

In [ ]:
ln_Response = filtered_base_data["Response"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(ln_Response, bins=50, kde=True, color="teal")
plt.title("Distribution of Response Lengths")
plt.xlabel("Length of Response")
plt.ylabel("Frequency")
plt.show()

In [ ]:
filtered_base_data = filtered_base_data[ln_Response <= 4000]

ln_Response = filtered_base_data["Response"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(ln_Response, bins=50, kde=True, color="teal")
plt.title("Distribution of Response Lengths")
plt.xlabel("Length of Response")
plt.ylabel("Frequency")
plt.show()

### My Data Answer Distribution

In [ ]:
ln_answer = filtered_my_data["answer"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(ln_answer, bins=50, kde=True, color="teal")
plt.title("Distribution of Answer Lengths")
plt.xlabel("Length of Answer")
plt.ylabel("Frequency")
plt.show()

In [ ]:
filtered_my_data = filtered_my_data[ln_answer <= 1500]

ln_answer = filtered_my_data["answer"].apply(len)
plt.figure(figsize=(10, 3))
sns.histplot(ln_answer, bins=50, kde=True, color="teal")
plt.title("Distribution of Answer Lengths")
plt.xlabel("Length of Answer")
plt.ylabel("Frequency")
plt.show()

## Preparing Model

### Load Model

In [ ]:
max_seq_length = 5020
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Meta-Llama-3-8B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # adjust the complexity and scaling of these adaptations.
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],  # layers of the model should be adapted.
    use_rslora=True,  # activates Rank-Stabilized LoRA.
    use_gradient_checkpointing="unsloth",  # storing only necessary computations.
    random_state = 32,
    loftq_config = None,
)

print(model.print_trainable_parameters())

### Prompt Formatting

In [ ]:
data_prompt = """Answer the user questions about Wordpress and help them.

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompt(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    texts = []
    for input_, output in zip(inputs, outputs):
        text = data_prompt.format(input_, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [ ]:
training_data = Dataset.from_pandas(filtered_my_data)
training_data = training_data.map(formatting_prompt, batched=True)

### Training

In [ ]:
trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=training_data,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=8,
        num_train_epochs=40,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="/src/output",
        seed=0,
    ),
)

In [ ]:
trainer.train()

### Test Fine-Tuned model for Inference

In [ ]:
text="How i can install plugin in Wordpress?"
model = FastLanguageModel.for_inference(model)

In [ ]:
inputs = tokenizer(
[
    data_prompt.format(
        #instructions
        text,
        #answer
        "",
    )
], return_tensors = "pt").to("cuda")

In [ ]:
outputs = model.generate(**inputs, max_new_tokens = 5020, use_cache = True)
answer=tokenizer.batch_decode(outputs)
answer = answer[0].split("### Response:")[-1]

print("Answer of the question is:", answer)

### Publish to HuggingFace

In [ ]:
os.environ["HF_TOKEN"] = "hugging face token key, you can create from your HF account."
model.push_to_hub("aminv/WordpressQA-Llama-3-8B-FT", use_auth_token=os.getenv("HF_TOKEN"))
tokenizer.push_to_hub("aminv/WordpressQA-Llama-3-8B-FT", use_auth_token=os.getenv("HF_TOKEN"))

### Save in Local

In [ ]:
model.save_pretrained("src/models/WordpressQA-Llama-3-8B-FT")
tokenizer.save_pretrained("src/models/WordpressQA-Llama-3-8B-FT")

### Load from Local

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "src/models/WordpressQA-Llama-3-8B-FT",
    max_seq_length = 5020,
    dtype = None,
    load_in_4bit = True)

# The End